[Lab README](README.md)

# Lab 5.1: Deploy the Lab 4 agent to AgentCore Runtime

The agent from Lab 4 runs on your machine. This notebook puts the same agent on
Amazon Bedrock AgentCore Runtime, invokes it four times over the network, and
leaves you an ARN. The retrieval tool does not change: `booking_agent.py`
imports `search_hotel_knowledge` from the same `workshop.hybrid_retrieval` that
Labs 2, 3 and 4 imported. That is the claim this notebook is here to test, and
the four smoke tests at the end are how it gets tested.

One thing does change. Locally, the reservation command was a Python function
the agent called in-process. Deployed, it is a Lambda behind an AgentCore
Gateway, and the agent discovers it as an MCP tool. The agent code is identical
either way, because both forms satisfy the same frozen five-field contract.

**This notebook creates billable AWS resources.** An ECR repository, a CodeBuild
project, and an AgentCore Runtime. `5.2_teardown.ipynb` deletes them, and
nothing else in the workshop does. Run it before you stop for the day.

**Prerequisites**

1. Lab 1 has built the graph, including the Cairo hero `Hotel` and the
   `max_guests` rule. The deployed agent reads that graph, not a copy of it.
2. `setup/provision_agentcore.py provision` has run. It creates the Gateway,
   the reservation Lambda, the Neo4j command secret, and the Runtime execution
   role, and writes three values into the repository-root `.env`. Step 1 checks
   for them and tells you what to run if they are missing.
3. AWS credentials with Bedrock model access in `AWS_REGION`, and Neo4j
   credentials in the repository-root `.env`.

Without credentials every live cell below skips and the notebook still runs
clean. That is how repository validation passes offline.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced (outside an AWS event): uncomment the line below first.
# ─────────────────────────────────────────────────────────────────────
# !pip install -r requirements.txt

print("Environment ready")

---
## Step 1: Confirm the managed boundary already exists

`provision_agentcore.py` builds the half of this deployment that this notebook
does not: the Gateway that fronts the reservation command, the Lambda behind it,
the secret that Lambda reads Neo4j from, and the IAM role the Runtime assumes.
Splitting it that way keeps this notebook to one job, deploying the agent, and
keeps the resource creation in a script you can read, re-run, and tear down from
a terminal.

Three values land in the repository-root `.env`:

| Value | What it is |
| --- | --- |
| `AGENTCORE_GATEWAY_URL` | The MCP endpoint the deployed agent discovers `create_reservation_request` from |
| `AGENTCORE_RUNTIME_ROLE_ARN` | The execution role the Runtime assumes |
| `NEO4J_COMMAND_SECRET_ID` | The Secrets Manager entry the reservation Lambda reads its Neo4j write credential from |

The Runtime itself reads Neo4j from environment variables rather than from a
secret, so its role grants no secret access at all. That is a deliberate
narrowing: the retrieval path is read-only and its credential is passed in as
container configuration, while the one path that writes to the graph reads a
credential the Runtime cannot see.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

import boto3
from dotenv import load_dotenv

# The repository-root `.env` is where provision_agentcore.py writes, and where
# the NEO4J_* values every other lab uses already live. A folder-local `.env`
# wins if a participant made one.
#
# The marker is setup/run_notebooks.py rather than README.md, because every lab
# folder has a README.md and the walk up would stop at the first one.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "setup" / "run_notebooks.py").is_file():
    if REPO_ROOT == REPO_ROOT.parent:
        raise FileNotFoundError(
            "Repository root not found. Run this notebook from 05-agentcore-deploy/."
        )
    REPO_ROOT = REPO_ROOT.parent
load_dotenv()
load_dotenv(REPO_ROOT / ".env")

REGION = os.environ.get("AWS_REGION", "us-east-1")
GATEWAY_URL = os.getenv("AGENTCORE_GATEWAY_URL", "").strip()
RUNTIME_ROLE_ARN = os.getenv("AGENTCORE_RUNTIME_ROLE_ARN", "").strip()
MODEL_ID = os.getenv("MODEL_ID", "us.anthropic.claude-sonnet-5")

# The deployed Runtime reads its read-only Neo4j connection from these, so they
# have to be forwarded as container environment variables at launch.
NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_VALUES = {name: os.getenv(name, "").strip() for name in NEO4J_ENV}

# This name is not free to change. workshop_cleanup.py derives the ECR
# repository and CodeBuild project names from it, so 5.2 can only tear down what
# 5.1 created if both files agree on it.
RUNTIME_NAME = "HotelBookingAgent"

# The teardown tag gate. workshop_cleanup.py deletes a resource only if it
# carries this exact key and value; a near-miss is the same as no tag at all.
# The three shapes below are not interchangeable, each service demands its own.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "stop-ai-agent-hallucinations"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                        # agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]       # ecr
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}] # codebuild

AWS_READY = boto3.Session().get_credentials() is not None
missing = [name for name, value in NEO4J_VALUES.items() if not value]
DEPLOY_READY = bool(GATEWAY_URL and RUNTIME_ROLE_ARN) and AWS_READY and not missing

print(f"region:            {REGION}")
print(f"model:             {MODEL_ID}")
print(f"runtime name:      {RUNTIME_NAME}")
print(f"AWS credentials:   {'found' if AWS_READY else 'NOT FOUND'}")
print(f"gateway URL:       {GATEWAY_URL or 'NOT SET'}")
print(f"runtime role:      {RUNTIME_ROLE_ARN or 'NOT SET'}")
print(f"Neo4j values:      {'all four present' if not missing else 'missing ' + ', '.join(missing)}")

if not DEPLOY_READY:
    print()
    print("Not ready to deploy. Every live cell below will skip.")
    if not (GATEWAY_URL and RUNTIME_ROLE_ARN):
        print("  Run the provisioning script first, from the repository root:")
        print("    uv run setup/provision_agentcore.py provision")
        print("  It is idempotent, and it creates real billable resources.")
    if missing:
        print(f"  Add to {REPO_ROOT / '.env'}: {', '.join(missing)}")
    if not AWS_READY:
        print("  Configure AWS credentials for the account that was provisioned.")
else:
    print()
    print("Ready to deploy.")

---
## Step 2: Give the image the shared package

`booking_agent.py` opens with an import that has to keep working inside a
container:

```python
from workshop.hybrid_retrieval import GROUNDING_INSTRUCTIONS, search_hotel_knowledge
```

That is the point of the whole deployment. The retrieval tool the Runtime serves
is the one Lab 2 built, imported rather than reimplemented, so there is no
second copy to drift.

It also means the image needs that package, and the package lives at the
repository root, outside this build context. A container build cannot reach up
out of its own context, so the package is built into a wheel here, next to the
Dockerfile, on every deploy. Building it rather than committing it means the
image always carries the source you have in front of you, including any edit you
made during Lab 4.

In [ ]:
DEPLOY_DIR = Path.cwd() / "deployment-tools"
if not DEPLOY_DIR.is_dir():
    DEPLOY_DIR = Path.cwd() if Path.cwd().name == "deployment-tools" else DEPLOY_DIR
if not DEPLOY_DIR.is_dir():
    raise FileNotFoundError(
        "deployment-tools/ not found. Run this notebook from 05-agentcore-deploy/."
    )

VENDOR_DIR = DEPLOY_DIR / "vendor"
PACKAGE_DIR = REPO_ROOT / "workshop"
REQUIREMENTS = DEPLOY_DIR / "agent_requirements.txt"

# Remove any earlier wheel first. A stale wheel from a previous version would
# still satisfy the pinned requirement line and would ship code you no longer
# have, which is exactly the kind of silent drift this lab is about.
for stale in VENDOR_DIR.glob("*.whl"):
    stale.unlink()
    print(f"Removed stale wheel: {stale.name}")

if shutil.which("uv") is None:
    raise RuntimeError(
        "uv is not on PATH, and it is what builds the wheel. Install it from "
        "https://docs.astral.sh/uv/ , or build the wheel by hand with "
        "`python -m build --wheel --outdir deployment-tools/vendor ../workshop`."
    )

subprocess.run(
    ["uv", "build", "--wheel", "--out-dir", str(VENDOR_DIR), str(PACKAGE_DIR)],
    check=True,
)

wheels = sorted(VENDOR_DIR.glob("*.whl"))
if len(wheels) != 1:
    raise RuntimeError(f"Expected exactly one wheel in {VENDOR_DIR}, found {wheels}")
wheel = wheels[0]

# agent_requirements.txt pins the wheel by filename rather than asking the
# resolver for a package called `workshop`, which would let it reach PyPI for
# something unrelated. The pin carries the version, so a version bump in
# workshop/pyproject.toml has to be reflected there too. Fail here, with the
# line to change, rather than inside a CodeBuild log ten minutes from now.
pinned = f"./vendor/{wheel.name}"
if pinned not in REQUIREMENTS.read_text(encoding="utf-8"):
    raise RuntimeError(
        f"{REQUIREMENTS.name} does not pin the wheel that was just built.\n"
        f"  built:    {pinned}\n"
        f"  Update that line in {REQUIREMENTS}, then re-run this cell."
    )

print(f"\nBuilt {wheel.name} ({wheel.stat().st_size // 1024} KB)")
print(f"Pinned in {REQUIREMENTS.name} as {pinned}")

---
## Step 3: Pre-flight cleanup

The starter toolkit writes a `.bedrock_agentcore.yaml` beside the entrypoint,
holding the runtime ID from the last deploy. A stale one makes this run try to
update a Runtime that teardown already deleted, so it goes first.

Note what this cell does **not** delete, and why:

- **The CodeBuild project stays.** `launch()` calls
  `create_or_update_project`, so an existing project is updated rather than
  colliding, and the deploy re-runs idempotently without a blind delete. A blind
  delete would also bypass the tag gate every other teardown path here honors.
- **The path is exact and local.** An earlier version globbed
  `~/.bedrock_agentcore*.yaml`. `HOME` is shared with every other AgentCore
  project on the machine, so running this workshop destroyed unrelated local
  config.

In [ ]:
# Configure and launch run from inside deployment-tools/, because that is the
# container build context. The starter toolkit uses the current directory as the
# build root: it honors the Dockerfile it finds there, and copies only that
# directory into the image. Run it from 05-agentcore-deploy/ instead and the
# toolkit would generate its own Dockerfile, ignore the one written for this
# agent, and ship the whole lab folder.
if Path.cwd() != DEPLOY_DIR:
    os.chdir(DEPLOY_DIR)
print(f"Build context: {Path.cwd()}")

# Pre-flight cleanup: remove the starter toolkit config from previous runs.
# The CodeBuild project is intentionally NOT deleted here. The starter toolkit's
# launch() calls create_or_update_project, which updates an existing project
# instead of raising ResourceAlreadyExistsException, so the deploy re-runs
# idempotently without a blind delete. A blind delete_project also bypassed the
# WorkshopResource tag gate that every other teardown path here honors.

# Delete starter toolkit config with old runtime ID.
# Remove only the config file THIS notebook's toolkit run writes, in THIS
# directory. A previous version globbed ~/.bedrock_agentcore*.yaml. HOME is
# shared with every other AgentCore project on the machine, so running this
# workshop destroyed unrelated local config. Exact path, current directory only.
local_cfg = os.path.join(os.getcwd(), ".bedrock_agentcore.yaml")
if os.path.exists(local_cfg):
    os.remove(local_cfg)
    print(f"Deleted stale config: {local_cfg}")

print("Pre-flight cleanup done")

---
## Step 4: Configure and launch

`launch()` ships the build context to CodeBuild, which builds an ARM64 image and
pushes it to ECR, then creates or updates the Runtime from that image. It takes
three to five minutes and prints nothing useful for most of them.

The environment variables are the whole configuration surface of the deployed
agent. `GATEWAY_URL` is how it finds the reservation command; without it,
`booking_agent.py` raises on the first invocation rather than answering as if it
had a tool it does not have. The four `NEO4J_*` values are the read-only
connection the retrieval tool uses. `MODEL_ID` keeps the deployed agent on the
same model the local one used, so a difference in behavior is a difference in
deployment and not a difference in model.

In [ ]:
if not DEPLOY_READY:
    print("Skipping launch: see Step 1.")
else:
    from bedrock_agentcore_starter_toolkit import Runtime

    print(f"Role:        {RUNTIME_ROLE_ARN}")
    print(f"Gateway URL: {GATEWAY_URL}")

    agent_runtime = Runtime()

    agent_runtime.configure(
        entrypoint="booking_agent.py",
        execution_role=RUNTIME_ROLE_ARN,
        auto_create_ecr=True,
        requirements_file="agent_requirements.txt",
        region=REGION,
        agent_name=RUNTIME_NAME,
        deployment_type="container",
        non_interactive=True,
    )

    print("\nLaunching agent (3-5 minutes)...")

    result = agent_runtime.launch(
        auto_update_on_conflict=True,
        env_vars={
            "AWS_REGION": REGION,
            "GATEWAY_URL": GATEWAY_URL,
            "MODEL_ID": MODEL_ID,
            **NEO4J_VALUES,
        },
    )

    RUNTIME_ARN = result.agent_arn
    RUNTIME_ID = RUNTIME_ARN.split("/")[-1] if RUNTIME_ARN else None
    print(f"\nAgent deployed: {RUNTIME_ARN}")

---
## Step 5: Tag the resources the toolkit created

The starter toolkit creates the ECR repository, the CodeBuild project and the
AgentCore Runtime on your behalf, and does not pass the workshop tag through.
`5.2_teardown.ipynb` deletes only tagged resources, so these three have to be
tagged now. Skip this cell and teardown will refuse to delete them, exit
non-zero, and you will keep paying for them.

In [ ]:
if not DEPLOY_READY:
    print("Skipping tagging: nothing was deployed.")
else:
    # --- Tag the resources the starter toolkit created ---
    # The toolkit creates the ECR repo, the CodeBuild project and the AgentCore
    # Runtime itself and does not forward tags, so they are tagged here,
    # immediately after deploy. Cleanup deletes only tagged resources; skip this
    # and teardown will refuse to remove them and will exit non-zero, leaving
    # billable infrastructure running.
    #
    # Every target below is addressed by EXACT name or by ARN. Nothing is
    # enumerated and nothing is prefix-matched. In particular the toolkit's
    # shared AmazonBedrockAgentCoreSDKCodeBuild-* IAM role is deliberately NOT
    # tagged: it is shared across projects, costs nothing, and tagging it would
    # make it eligible for deletion. Deleting roles by name shape once destroyed
    # five unrelated roles in this account.

    ecr_client = boto3.client("ecr", region_name=REGION)
    codebuild_client = boto3.client("codebuild", region_name=REGION)
    agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

    ECR_REPO = f"bedrock-agentcore-{RUNTIME_NAME.lower()}"
    CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

    # ECR repository: resourceArn=, but capitalised {Key, Value} members
    try:
        repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
        print(f"Tagged ECR repository: {ECR_REPO}")
    except ecr_client.exceptions.RepositoryNotFoundException:
        print(f"ECR repository not found (nothing to tag): {ECR_REPO}")

    # CodeBuild project: lowercase key/value members, applied via update_project.
    # update_project REPLACES the whole tag set, so merge rather than clobber
    # whatever the starter toolkit put there.
    projects = codebuild_client.batch_get_projects(names=[CB_PROJECT])["projects"]
    if projects:
        merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
        codebuild_client.update_project(name=CB_PROJECT, tags=merged + WORKSHOP_TAGS_KV_LOWER)
        print(f"Tagged CodeBuild project: {CB_PROJECT}")
    else:
        print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT}")

    # AgentCore Runtime: tag by the ARN the launch returned
    if not RUNTIME_ARN:
        raise RuntimeError("RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
    agentcore.tag_resource(resourceArn=RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)
    print(f"Tagged AgentCore Runtime: {RUNTIME_ARN}")

    # Verify rather than trust: read the tags back.
    runtime_tags = agentcore.list_tags_for_resource(resourceArn=RUNTIME_ARN).get("tags", {})
    if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
        raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
    print("\nAll toolkit-created resources tagged and verified.")

---
## Step 6: Four smoke tests

These are the four behaviors Labs 2, 3 and 4 established, asked again over the
network against the deployed Runtime. They are the same four because that is the
test: deployment is supposed to change where the agent runs and nothing about
what it does.

| # | Question | What passing looks like |
| --- | --- | --- |
| 1 | The hero question | Grounded amenities and rating, sourced from the graph |
| 2 | The availability question | Abstention. The graph holds no live availability |
| 3 | A 15-guest reservation request | Rejected, `max_guests_exceeded`, nothing written |
| 4 | The same request delivered twice | One record, `duplicate` on the second delivery |

Tests 3 and 4 write to your graph. They create one `ReservationRequest` node
linked to the Cairo hero hotel, which is the same node Lab 4 created locally.

In [ ]:
import json
import uuid
from datetime import date, timedelta

from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from workshop.graph_setup import HERO_NAME

# One caller-created UUID, reused for every delivery of the same reservation
# request. It is the idempotency key and the correlation identifier across
# Runtime, Gateway, the Lambda, and the CloudWatch log lines for all three.
REQUEST_ID = str(uuid.uuid4())

# Relative to today, never a hardcoded date. A fixed future date rots into the
# past and silently flips a passing check-in into a failing one.
CHECK_IN = (date.today() + timedelta(days=30)).isoformat()
CHECK_OUT = (date.today() + timedelta(days=32)).isoformat()


def ask(prompt, request_id=None, session_id=None):
    """Invoke the deployed Runtime once and print what came back."""
    payload = {"prompt": prompt}
    if request_id is not None:
        payload["request_id"] = request_id

    client = boto3.client("bedrock-agentcore", region_name=REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=RUNTIME_ARN,
        runtimeSessionId=session_id or str(uuid.uuid4()),
        payload=json.dumps(payload).encode("utf-8"),
        qualifier="DEFAULT",
    )
    result = json.loads(response["response"].read())

    print(f"Q: {prompt}\n")
    print(f"A: {result.get('response')}\n")
    print(f"tools used: {result.get('tools_used') or 'none'}")
    return result


if DEPLOY_READY:
    print(f"request_id: {REQUEST_ID}")
    print(f"stay:       {CHECK_IN} to {CHECK_OUT}")
else:
    print("Skipping smoke tests: nothing was deployed.")

In [ ]:
# Test 1: the hero question. Grounded retrieval over the network.
if not DEPLOY_READY:
    print("Skipped.")
else:
    ask(f"What amenities and guest rating does {HERO_NAME} have?")

In [ ]:
# Test 2: the availability question. The graph holds no live availability, so
# the correct answer is to decline rather than to invent one.
if not DEPLOY_READY:
    print("Skipped.")
else:
    ask(f"Does {HERO_NAME} guarantee room availability next weekend?")

In [ ]:
# Test 3: an over-limit reservation request. The max_guests rule lives in the
# graph and the command reads it, so the rejection happens inside the same
# boundary as the write. Nothing is written.
if not DEPLOY_READY:
    print("Skipped.")
else:
    ask(
        f"Find {HERO_NAME} and create a reservation request for "
        f"{OVER_LIMIT_GUESTS} guests, check-in {CHECK_IN}, check-out {CHECK_OUT}.",
        request_id=REQUEST_ID,
    )

In [ ]:
# Test 4: a corrected request within the limit, delivered twice with the same
# request_id. The second delivery returns the existing record rather than
# creating a second one. Idempotence is the command's job, not the model's.
if not DEPLOY_READY:
    print("Skipped.")
else:
    prompt = (
        f"Find {HERO_NAME} and create a reservation request for "
        f"{MAX_GUESTS} guests, check-in {CHECK_IN}, check-out {CHECK_OUT}."
    )
    print("=== first delivery ===")
    ask(prompt, request_id=REQUEST_ID)
    print("\n=== second delivery, same request_id ===")
    ask(prompt, request_id=REQUEST_ID)

---
## What is running now

An AgentCore Runtime, an ECR repository holding its image, and a CodeBuild
project that built it. All three are tagged, which is the only reason
`5.2_teardown.ipynb` can delete them.

## Where the request went

The `request_id` you generated above appears in the Runtime logs, in the
Gateway's record of the tool call, in the reservation Lambda's logs, and on the
`ReservationRequest` node in your graph. One identifier, created by the caller,
carried end to end. That is what makes a deployed agent's behavior something you
can audit after the fact rather than something you have to reproduce.

Runtime logs are in CloudWatch under `/aws/bedrock-agentcore/runtimes/`.

## Next

- **`5.3_agentcore_walkthrough.ipynb`** (optional) works through the rejection,
  the correction, the graph inspection, and the log correlation one at a time. It
  reads `AGENT_RUNTIME_ARN` from the environment, so export the ARN printed
  above if you want to run it in a fresh kernel.
- **`5.2_teardown.ipynb`** deletes what this notebook created. Run it before you
  stop. It is the only thing in the workshop that does.